# 05 - Model Training
Trains Linear Regression, Decision Tree, Random Forest, Gradient Boosting, SVR, XGBoost, and LightGBM to predict `temperature_celsius`.

**Leakage note:** engineered features that are deterministic functions of *today's* temperature (heat index, wind chill, feels-like, etc.) are excluded from the feature matrix — see `src/train.py::get_feature_matrix` for details. Only past-looking lag/rolling temperature features are kept.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
import pandas as pd
from src import config
from src.train import get_feature_matrix, train_test_split_data, train_all_models, build_ensemble

df = pd.read_csv(config.FEATURED_DATA_FILE, parse_dates=[config.DATETIME_COL])
X, y, feature_cols = get_feature_matrix(df)
print('Feature matrix:', X.shape)
print('Features:', feature_cols)

Feature matrix: (12000, 45)
Features: ['latitude', 'longitude', 'wind_mph', 'wind_kph', 'wind_degree', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'moon_illumination', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'day_of_year', 'humidity_index', 'temperature_celsius_lag_1', 'temperature_celsius_lag_2', 'temperature_celsius_lag_3', 'temperature_celsius_lag_7', 'temperature_celsius_rollmean_3', 'temperature_celsius_rollstd_3', 'temperature_celsius_rollmean_7', 'temperature_celsius_rollstd_7', 'temperature_celsius_rollmean_14', 'temperature_celsius_rollstd_14', 'wind_pressure_interaction']


In [3]:
X_train, X_test, y_train, y_test = train_test_split_data(X, y)
print('Train:', X_train.shape, ' Test:', X_test.shape)

Train: (9600, 45)  Test: (2400, 45)


## Train all models

In [4]:
fitted_models = train_all_models(X_train, y_train)
list(fitted_models.keys())

2026-07-03 05:37:26 | src.train | INFO | Training LinearRegression ...


2026-07-03 05:37:26 | src.train | INFO | Saved LinearRegression -> /home/claude/Weather-Trend-Forecasting/models/LinearRegression.pkl


2026-07-03 05:37:26 | src.train | INFO | Training DecisionTree ...


2026-07-03 05:37:27 | src.train | INFO | Saved DecisionTree -> /home/claude/Weather-Trend-Forecasting/models/DecisionTree.pkl


2026-07-03 05:37:27 | src.train | INFO | Training RandomForest ...


...

2026-07-03 05:38:58 | src.train | INFO | Saved XGBoost -> /home/claude/Weather-Trend-Forecasting/models/XGBoost.pkl


2026-07-03 05:38:58 | src.train | INFO | Training LightGBM ...


2026-07-03 05:39:00 | src.train | INFO | Saved LightGBM -> /home/claude/Weather-Trend-Forecasting/models/LightGBM.pkl


2026-07-03 05:39:00 | src.train | INFO | train_all_models finished in 93.42s


['LinearRegression',
 'DecisionTree',
 'RandomForest',
 'GradientBoosting',
 'SVR',
 'XGBoost',
 'LightGBM']

## Build a simple weighted-average ensemble

In [5]:
import joblib
ensemble_pred = build_ensemble(fitted_models, X_test)
joblib.dump(fitted_models, config.MODELS_DIR / 'all_models_bundle.pkl')
ensemble_pred[:10]

array([ 5.70434087, 18.35144851, 26.58553846,  9.47966729, 11.56764971,
        3.94980205, 19.80773146,  2.02327254,  7.06503246, 10.91983296])